In [1]:
suppressPackageStartupMessages({
    library(data.table)
    library(Matrix)
    library(dplyr)
    library(SingleCellExperiment)
    library(batchelor)
    library(scuttle)
    library(scater)
    library(scran)
    library(BiocParallel)
})

In [2]:
getwd()

[1] "/rds/project/rds-SDzz0CATGms/users/bt392/atlasses/extended/code"

In [3]:
io = list()
io$main = '/rds/project/rds-SDzz0CATGms/users/bt392/atlasses/extended'
io$sce = file.path(io$main, '/embryo_sce.rds')
io$outdir = file.path(io$main, 'endothelium')
dir.create(io$outdir, recursive=T)

Warning message in dir.create(io$outdir, recursive = T):
“'/rds/project/rds-SDzz0CATGms/users/bt392/atlasses/extended/endothelium' already exists”


In [4]:
sce = readRDS(io$sce)

In [5]:
meta = as.data.table(colData(sce))

In [6]:
unique(meta$embryo_version)

[1] Original  Extension
Levels: Extension Original

In [7]:
unique(meta$celltype_extended_atlas)

[1] Epiblast                                      
 [2] Primitive Streak                              
 [3] ExE ectoderm                                  
 [4] Visceral endoderm                             
 [5] ExE endoderm                                  
 [6] Non-neural ectoderm                           
 [7] Nascent mesoderm                              
 [8] Ectoderm                                      
 [9] Blood progenitors                             
[10] Paraxial mesoderm                             
[11] Caudal epiblast                               
[12] Lateral plate mesoderm                        
[13] Intermediate mesoderm                         
[14] Cardiopharyngeal progenitors SHF              
[15] PGC                                           
[16] Mesenchyme                                    
[17] Haematoendothelial progenitors                
[18] Gut tube                                      
[19] Cardiomyocytes FHF 1                          
[20] Allantois endothelium                         
[21] Node                                          
[22] Erythroid                                     
[23] Embryo proper endothelium                     
[24] Anterior somitic tissues                      
[25] Cranial mesoderm                              
[26] Amniotic ectoderm                             
[27] Parietal endoderm                             
[28] Anterior Primitive Streak                     
[29] Presomitic mesoderm                           
[30] Allantois                                     
[31] Limb ectoderm                                 
[32] EMP                                           
[33] Anterior cardiopharyngeal progenitors         
[34] Pharyngeal mesoderm                           
[35] Limb mesoderm                                 
[36] Midgut                                        
[37] Venous endothelium                            
[38] Pharyngeal endoderm                           
[39] Hindgut                                       
[40] Notochord                                     
[41] Epicardium                                    
[42] Posterior somitic tissues                     
[43] Optic vesicle                                 
[44] Somitic mesoderm                              
[45] Hindbrain floor plate                         
[46] Caudal mesoderm                               
[47] Thyroid primordium                            
[48] Dermomyotome                                  
[49] Cardiomyocytes SHF 1                          
[50] Placodal ectoderm                             
[51] NMPs                                          
[52] Dorsal spinal cord progenitors                
[53] Spinal cord progenitors                       
[54] Midbrain progenitors                          
[55] Ventral forebrain progenitors                 
[56] Hindbrain neural progenitors                  
[57] Cardiomyocytes FHF 2                          
[58] Cardiopharyngeal progenitors FHF              
[59] Late dorsal forebrain progenitors             
[60] Migratory neural crest                        
[61] Ventral hindbrain progenitors                 
[62] Midbrain/Hindbrain boundary                   
[63] Foregut                                       
[64] Surface ectoderm                              
[65] YS mesothelium                                
[66] Embryo proper mesothelium                     
[67] Sclerotome                                    
[68] Otic placode                                  
[69] Dorsal hindbrain progenitors                  
[70] Cardiomyocytes SHF 2                          
[71] YS endothelium                                
[72] NMPs/Mesoderm-biased                          
[73] Kidney primordium                             
[74] Neural tube                                   
[75] Endocardium                                   
[76] MEP                                           
[77] Dorsal midbrain neurons                     

In [8]:
ct_keep = c('Allantois endothelium', 
            'Embryo proper endothelium',
            'EMP',
            'Venous endothelium',
            'YS endothelium',
           'Endocardium',
           'Blood progenitors')

In [9]:
meta_endo = meta[celltype_extended_atlas %in% ct_keep]

In [10]:
unique(meta_endo$embryo_version)

[1] Original  Extension
Levels: Extension Original

In [11]:
sce_endo = sce[,as.character(meta_endo$cell)]

In [12]:
# Add gene names
genes = fread('/rds/project/rds-SDzz0CATGms/users/bt392/atlasses/Mmusculus_genes_BioMart.87.txt')
rownames(sce_endo) = genes[match(rownames(sce_endo), genes$ens_id), symbol] 

In [13]:
colnames(colData(sce_endo))

[1] "cell"                                           
 [2] "sample"                                         
 [3] "embryo_version"                                 
 [4] "stage"                                          
 [5] "stage_mixed_gastrulation_mapped"                
 [6] "somite_count"                                   
 [7] "anatomy"                                        
 [8] "S_score"                                        
 [9] "G2M_score"                                      
[10] "phase"                                          
[11] "r_louvain_clus"                                 
[12] "r_louvain_subclus"                              
[13] "louvain"                                        
[14] "leiden"                                         
[15] "celltype_PijuanSala2019"                        
[16] "celltype_PijuanSala2019_E85mapped"              
[17] "celltype_PijuanSala2019_E85mapped_WOTdescendant"
[18] "celltype_extended_atlas"                        
[19] "sizeFactor"

In [14]:
colData(sce_endo)$sizeFactor = as.numeric(colData(sce_endo)$sizeFactor)
sizeFactors(sce_endo) = colData(sce_endo)$sizeFactor

In [15]:
# Log normalisation
sce_endo = logNormCounts(sce_endo)

In [16]:
saveRDS(sce_endo, file.path(io$outdir, 'endo_sce.rds'))

In [17]:
sce_endo

In [18]:
head(sce_endo)

class: SingleCellExperiment 
dim: 6 22413 
metadata(0):
assays(2): counts logcounts
rownames(6): Xkr4 Gm1992 ... Sox17 Gm37323
rowData names(0):
colnames(22413): cell_363 cell_382 ... ext_cell_351834 ext_cell_351864
colData names(19): cell sample ... celltype_extended_atlas sizeFactor
reducedDimNames(0):
mainExpName: NULL
altExpNames(0):